# The Grounding Gap in Turkish

Extension of Chlapanis, Menis Mastromichalakis and Papadimitriou, *The Grounding Gap: How LLMs Anchor the Meaning of Abstract Concepts Differently from Humans* (arXiv 2605.08837). Code: [github.com/odychlapanis/grounding-gap](https://github.com/odychlapanis/grounding-gap).

**What this notebook does.** Runs the paper's Experiment 1 (Harpaintner et al. 2018, 293 abstract words) in English and in Turkish on the same model, **Gemini 2.5 Flash-Lite**, which is both one of the paper's 21 models (Mean r = 0.348) and its canonical coder. Every property in both languages is coded with the paper's own English coding prompt, so generation language is the only thing that changes between the two arms.

**Three comparisons.**
1. Turkish vs English, same model, both against the Harpaintner human norms. The norms come from German speakers, so English is already a translation of the original stimuli; neither language is the privileged one.
2. Within Turkish: does translating a concept into a transparent native word (*izlenim*, from *iz* "footprint") shift its profile more than translating it into an opaque loanword (*adalet*)? Each concept's English profile is its own baseline.
3. A correction to the released English leaderboard (optional, section 7).

**Before you start.**
- Put a Gemini API key in Colab secrets as `GEMINI_API_KEY` (key icon in the left sidebar). Get one at [aistudio.google.com/apikey](https://aistudio.google.com/apikey).
- **Gemini 2.5 Flash-Lite retires on the Gemini API no earlier than 16 October 2026.** Run this before then, or switch `MODEL` to `gemini-3.1-flash-lite`, which is also on the paper's leaderboard (Mean r = 0.223).
- Free tier works but is slow (about 15 requests a minute). The full run is roughly 3,500 calls. With billing enabled and a spending cap set, the whole study costs well under one US dollar and finishes in minutes. Set `FREE_TIER` accordingly.
- Every step is resumable and, with `USE_DRIVE = True`, saved to Google Drive. If Colab disconnects, reconnect and run all cells again; finished work is skipped.

In [ ]:
# ---- configuration ----
MODEL      = "gemini-2.5-flash-lite"   # generator; the paper's rank-3 model
CODER      = "gemini-2.5-flash-lite"   # the paper's canonical Experiment 1 coder
SHIPPED_DIR = "gemini-2.5-flash-lite"  # the paper's released runs for the same model (English sanity check)

N_RUNS     = 3        # the paper used 10; 3 gives split-half reliability around 0.74 for this model
SUBSET_N   = 30       # start with 30 words as a pilot, then set to None for all 293
FREE_TIER  = True     # False once billing is enabled with a spending cap
USE_DRIVE  = True     # save everything to Google Drive so a dead session loses nothing
SEED       = 0

RPM, CONCURRENCY = (14, 1) if FREE_TIER else (1500, 8)

In [ ]:
# ---- setup ----
import os, sys, subprocess
if not os.path.exists("/content/grounding-gap"):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/odychlapanis/grounding-gap.git", "/content/grounding-gap"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "openai>=1.30", "pandas", "numpy", "scipy", "tqdm", "matplotlib"], check=True)
REPO = "/content/grounding-gap/property_generation_experiments"

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT = "/content/drive/MyDrive/grounding_gap_tr"
else:
    OUT = "/content/gg_tr_out"
os.makedirs(OUT, exist_ok=True)

try:
    from google.colab import userdata
    API_KEY = userdata.get("GEMINI_API_KEY")
except Exception:
    import getpass
    API_KEY = getpass.getpass("Gemini API key: ")
assert API_KEY, "Add GEMINI_API_KEY to Colab secrets first"
print("repo:", REPO, "\noutput:", OUT)

In [ ]:
%%writefile /content/gg_tr.py
"""Cross-language extension of the Grounding Gap property-generation experiment.

Runs Harpaintner-style property generation in English and Turkish on the same
model, codes every property with the paper's own coder, and compares the two
languages against each other and against the published human norms.

Designed for Google Colab: every step is resumable, so a dead session loses at
most the call that was in flight.
"""
from __future__ import annotations

import ast
import csv
import os
import random
import re
import threading
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from typing import Callable, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
from scipy import stats

CATS = ["SM", "IS_E", "SC", "VA"]
CAT_LABELS = {"SM": "Sensorimotor", "IS_E": "Internal state", "SC": "Social", "VA": "Verbal association"}

# ---------------------------------------------------------------------------
# Prompts
# ---------------------------------------------------------------------------

# Turkish generation prompt. A direct translation of the repo's
# data/experiment_1/generation_prompt.txt, keeping the same worked example.
TR_GENERATION_PROMPT = (
    "Bir kelime üretme görevi yapacaksınız. Verilen '{word}' kelimesi için, aklınıza "
    "olabildiğince kendiliğinden gelen dört farklı çağrışım/durum/özellik (bundan sonra "
    "'özellikler' olarak anılacak) üretin. Eş anlamlı kelimeler kullanmayın. Cevabınız "
    "yalnızca kelimeyi ve dört özelliği içermeli ve şu örnekteki gibi biçimlendirilmelidir:\n"
    "kelime: sempati\n"
    "özellikler: sarılma, arkadaşlar, neşe, güneş"
)

# Optional robustness check only. The main design keeps the coder prompt in
# English for both languages, so that generation language is the only thing
# that changes between the two arms.
TR_CODING_PROMPT = (
    "Size soyut bir kelime ve bu kelimeye ait dört özellik verilecek. Her özelliği aşağıdaki "
    "beş kategoriden birine sınıflandırmanız gerekiyor.\n\n"
    "Duyusal-motor özellik: Duyularımızla deneyimlenebilen bir özellik. Soyut kavramın anlamını "
    "betimler ya da soyut kavram bu özelliğe uygulanabilir.\n"
    "Sosyal yapı: Farklı kişilerin bir arada bulunmasını betimleyen ya da en az iki farklı kişi "
    "arasında bir etkileşimi ima eden bir özellik veya durum.\n"
    "İçsel durum ve duygu: İçsel, bilişsel süreçleri (ör. motivasyon, duygu, irade) yansıtan bir "
    "özellik veya durum.\n"
    "Çağrışım: Soyut kavramı betimlemeyen, ama onunla tematik ya da sembolik olarak ilişkili olan "
    "bir özellik veya durum.\n"
    "Diğer soyut kavram: Soyut kavramı betimleyen ya da soyut kavramın uygulanabileceği soyut bir "
    "özellik.\n\n"
    "Her özelliği ve kategorisini aşağıdaki örnekteki gibi yazın (Soyut kelime: sempati, "
    "Özellikler: sarılma, arkadaşlar, neşe, güneş).\n"
    "sarılma: Duyusal-motor özellik\n"
    "arkadaşlar: Sosyal yapı\n"
    "neşe: İçsel durum ve duygu\n"
    "güneş: Çağrışım\n\n"
    "Soyut kelime: {word}\n"
    "Özellikler: {properties}"
)

EN_LABEL_MAP = {  # identical to the repo's EXP1_LABEL_MAP
    "sensorimotor feature": "SM",
    "internal state and emotion": "IS_E",
    "social constellation": "SC",
    "association": "VA",
    "other abstract concept": "VA",
}
TR_LABEL_MAP = {
    "duyusal-motor özellik": "SM",
    "duyusal motor özellik": "SM",
    "içsel durum ve duygu": "IS_E",
    "sosyal yapı": "SC",
    "diğer soyut kavram": "VA",
    "çağrışım": "VA",
}

# ---------------------------------------------------------------------------
# Parsing
# ---------------------------------------------------------------------------

_PROP_RE = re.compile(r"(?:properties|özellikler|ozellikler)\s*:", re.IGNORECASE)
_ITEM_RE = re.compile(r"^\s*(?:\d+[\.\)]|[-*•])\s*(.+?)\s*$")
_ERROR_RE = re.compile(r"^\s*error\b|\b50[0-9]\b.*(unavailable|server error)|rate limit", re.IGNORECASE)


def _split(line: str) -> List[str]:
    return [p.strip().strip(".").strip() for p in line.split(",") if p.strip().strip(".").strip()]


def parse_properties(text: str, word: Optional[str] = None) -> Tuple[List[str], str]:
    """Properties from a Harpaintner-style answer, English or Turkish.

    Returns (properties, mode). mode records which rule fired, so format drift
    between languages is visible instead of silently becoming missing data:
      label      'properties: a, b, c, d' (the format the prompt asks for)
      word_colon 'anger: a, b, c, d'      (the format the repo's parser misses)
      list       numbered or bulleted lines
      bare       a single comma-separated line
      error      an API error string stored as a response
      none       nothing recoverable
    """
    text = (text or "").replace("**", "").strip()
    if not text or _ERROR_RE.search(text[:200]):
        return [], "error"
    m = _PROP_RE.search(text)
    if m:
        rest = text[m.end():]
        for line in rest.splitlines():
            if line.strip():
                if _ITEM_RE.match(line):
                    break
                return _split(line), "label"
        items = [_ITEM_RE.match(l).group(1) for l in rest.splitlines() if _ITEM_RE.match(l)]
        if items:
            return [i.strip().strip(".") for i in items], "list"
        return [], "none"
    lines = [l for l in text.splitlines() if l.strip()]
    for l in lines:
        if ":" in l:
            head, tail = l.split(":", 1)
            if tail.count(",") >= 2 and (word is None or head.strip().lower().strip("'\"") in
                                         (word.lower(), "word", "kelime")):
                return _split(tail), "word_colon"
    items = [_ITEM_RE.match(l).group(1) for l in lines if _ITEM_RE.match(l)]
    if len(items) >= 2:
        return [i.strip().strip(".") for i in items], "list"
    commas = [l for l in lines if l.count(",") >= 2 and ":" not in l]
    if commas:
        return _split(commas[-1]), "bare"
    return [], "none"


def _tr_lower(s: str) -> str:
    return s.replace("I", "ı").replace("İ", "i").lower()


def parse_codes(text: str, label_map: Dict[str, str]) -> Tuple[List[str], List[str]]:
    """Map each '<property>: <Category>' line to a category.

    Returns (codes, raw_labels). raw_labels keeps the coder's own wording so the
    'other abstract concept' share can be recovered later, which the original
    release does not make easy.
    """
    codes, raws = [], []
    # longest keys first so 'other abstract concept' never matches as 'association'
    keys = sorted(label_map, key=len, reverse=True)
    for line in (text or "").strip().splitlines():
        if ":" not in line:
            continue
        tail = line.rsplit(":", 1)[1].strip()
        # try both case-foldings: plain lower() for English labels, Turkish-aware
        # lower for Turkish ones (plain lower() maps 'I' to 'i', which breaks
        # 'İçsel'; Turkish lower maps 'I' to 'ı', which breaks 'Internal')
        cands = (tail.lower(), _tr_lower(tail))
        for k in keys:
            if any(k in c for c in cands):
                codes.append(label_map[k])
                raws.append(k)
                break
    return codes, raws


def frequencies(codes: List[str]) -> List[float]:
    n = len(codes)
    return [codes.count(c) / n for c in CATS] if n else [0.0] * len(CATS)


# ---------------------------------------------------------------------------
# Client with rate limiting, retries and usage accounting
# ---------------------------------------------------------------------------

class Limiter:
    def __init__(self, rpm: float):
        self.interval = 60.0 / rpm if rpm else 0.0
        self.lock = threading.Lock()
        self.next_t = 0.0

    def wait(self):
        with self.lock:
            now = time.monotonic()
            t = max(now, self.next_t)
            self.next_t = t + self.interval
        delay = t - time.monotonic()
        if delay > 0:
            time.sleep(delay)


class Usage:
    def __init__(self):
        self.lock = threading.Lock()
        self.calls = 0
        self.prompt_tokens = 0
        self.completion_tokens = 0
        self.failures = 0

    def add(self, p, c):
        with self.lock:
            self.calls += 1
            self.prompt_tokens += p or 0
            self.completion_tokens += c or 0

    def cost(self, in_per_m=0.10, out_per_m=0.40):
        return self.prompt_tokens / 1e6 * in_per_m + self.completion_tokens / 1e6 * out_per_m


def make_gemini_query(api_key: str, rpm: float = 14, max_retries: int = 8,
                      base_url: str = "https://generativelanguage.googleapis.com/v1beta/openai/"):
    """Query function with the same signature the repo's clients use."""
    from openai import OpenAI

    client = OpenAI(api_key=api_key, base_url=base_url, max_retries=0)
    limiter, usage = Limiter(rpm), Usage()

    def query(prompt: str, model_name: str, temperature: float = 1.0, max_tokens: int = 1024,
              **_ignored) -> str:
        last = None
        for attempt in range(max_retries):
            limiter.wait()
            try:
                r = client.chat.completions.create(
                    model=model_name,
                    messages=[{"role": "user", "content": prompt}],
                    temperature=temperature,
                    max_tokens=max_tokens,
                )
                u = getattr(r, "usage", None)
                usage.add(getattr(u, "prompt_tokens", 0), getattr(u, "completion_tokens", 0))
                return r.choices[0].message.content or ""
            except Exception as e:  # rate limits, 5xx, transient network
                last = e
                msg = str(e)
                fatal = any(s in msg for s in ("API key not valid", "PERMISSION_DENIED", "404", "not found"))
                if fatal:
                    raise
                time.sleep(min(60, 2 ** attempt + random.random()))
        usage.failures += 1
        raise RuntimeError(f"gave up after {max_retries} attempts: {last}")

    query.usage = usage
    query.client = client
    return query


# ---------------------------------------------------------------------------
# Resumable runners
# ---------------------------------------------------------------------------

def _done_keys(path: str) -> set:
    if not os.path.exists(path) or os.path.getsize(path) == 0:
        return set()
    return set(pd.read_csv(path, usecols=["key"])["key"].astype(str))


def _pool(items, fn, concurrency, desc):
    try:
        from tqdm.auto import tqdm
    except Exception:  # pragma: no cover
        tqdm = lambda x, **k: x
    errors = 0
    with ThreadPoolExecutor(max_workers=concurrency) as ex:
        futs = [ex.submit(fn, it) for it in items]
        for f in tqdm(as_completed(futs), total=len(futs), desc=desc):
            try:
                f.result()
            except Exception as e:
                errors += 1
                if errors == 1:
                    import traceback
                    traceback.print_exc()
                elif errors <= 5:
                    print(f"  error: {e}")
    if errors:
        print(f"  {errors} item(s) failed; re-run this cell to retry only those")
    return errors


def generate(stimuli: List[Tuple[str, str]], prompt: str, model: str, out_path: str,
             query: Callable, concurrency: int = 1, temperature: float = 1.0) -> int:
    """stimuli: list of (key, word shown to the model). key is the English norm word."""
    done = _done_keys(out_path)
    todo = [s for s in stimuli if s[0] not in done]
    if not todo:
        print(f"  complete: {os.path.basename(out_path)}")
        return 0
    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    new = not os.path.exists(out_path) or os.path.getsize(out_path) == 0
    lock = threading.Lock()
    with open(out_path, "a", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        if new:
            w.writerow(["key", "stimulus", "response", "properties", "parse_mode"])
            f.flush()

        def one(item):
            key, word = item
            resp = query(prompt.replace("{word}", word), model_name=model, temperature=temperature)
            props, mode = parse_properties(resp, word)
            with lock:
                w.writerow([key, word, resp, props, mode])
                f.flush()

        return _pool(todo, one, concurrency, os.path.basename(out_path))


def code(gen_path: str, coding_prompt: str, coder: str, out_path: str, query: Callable,
         label_map: Dict[str, str], concurrency: int = 1) -> int:
    gen = pd.read_csv(gen_path)
    done = _done_keys(out_path)
    todo = [r for _, r in gen.iterrows() if str(r["key"]) not in done]
    if not todo:
        print(f"  complete: {os.path.basename(out_path)}")
        return 0
    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    new = not os.path.exists(out_path) or os.path.getsize(out_path) == 0
    lock = threading.Lock()
    with open(out_path, "a", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        if new:
            w.writerow(["key", "stimulus", "properties", "codes", "raw_labels", "frequencies"])
            f.flush()

        def one(r):
            try:
                props = ast.literal_eval(str(r["properties"]))
            except Exception:
                props = []
            if not props:
                with lock:
                    w.writerow([r["key"], r["stimulus"], [], [], [], [0.0] * 4])
                    f.flush()
                return
            p = coding_prompt.replace("{word}", str(r["stimulus"])).replace("{properties}", ", ".join(props))
            resp = query(p, model_name=coder, temperature=0.0)
            codes, raws = parse_codes(resp, label_map)
            with lock:
                w.writerow([r["key"], r["stimulus"], props, codes, raws, frequencies(codes)])
                f.flush()

        return _pool(todo, one, concurrency, os.path.basename(out_path))


# ---------------------------------------------------------------------------
# Loading and aggregation
# ---------------------------------------------------------------------------

def load_coded(paths: List[str]) -> pd.DataFrame:
    """Long table: one row per (run, key) with the four frequencies."""
    rows = []
    for i, p in enumerate(sorted(paths)):
        df = pd.read_csv(p)
        for _, r in df.iterrows():
            try:
                fr = ast.literal_eval(str(r["frequencies"]))
                codes = ast.literal_eval(str(r["codes"]))
            except Exception:
                continue
            if not codes:  # no usable coding: drop rather than count as zeros
                continue
            rows.append(dict(run=i, key=str(r["key"]), **dict(zip(CATS, fr))))
    return pd.DataFrame(rows)


def profile(long: pd.DataFrame) -> pd.DataFrame:
    """Per-word mean frequency vector across runs, indexed by English key."""
    return long.groupby("key")[CATS].mean()


def load_shipped_profile(repo_exp_dir: str, model_dir: str, as_released: bool = True) -> Optional[pd.DataFrame]:
    """English profile from the paper's own released runs for the same model, if shipped.

    as_released=True reproduces evaluate.py exactly, including the all-zero rows
    left by answers its parser could not read. as_released=False drops them.
    """
    import glob
    paths = sorted(glob.glob(os.path.join(repo_exp_dir, "coded_generations", "exp1", model_dir, "*.csv")))
    if not paths:
        return None
    rows = []
    for p in paths:
        df = pd.read_csv(p)
        for _, r in df.iterrows():
            try:
                fr = ast.literal_eval(str(r["frequencies"]))
            except Exception:
                continue
            if len(fr) == 4 and (as_released or sum(fr) > 0):
                rows.append(dict(key=str(r["word"]).strip(), **dict(zip(CATS, fr))))
    return pd.DataFrame(rows).groupby("key")[CATS].mean()


# ---------------------------------------------------------------------------
# Statistics
# ---------------------------------------------------------------------------

def mean_r(prof: pd.DataFrame, human: pd.DataFrame, keys=None) -> Dict[str, float]:
    keys = list(prof.index.intersection(human.index) if keys is None else keys)
    out = {c: stats.pearsonr(prof.loc[keys, c], human.loc[keys, c])[0] for c in CATS}
    out["mean"] = float(np.mean([out[c] for c in CATS]))
    out["n"] = len(keys)
    return out


def paired_bootstrap(en: pd.DataFrame, tr: pd.DataFrame, human: pd.DataFrame,
                     keys: List[str], n_boot: int = 2000, seed: int = 0) -> Dict[str, Tuple]:
    """Resample words (paired across languages). CIs for Turkish minus English."""
    rng = np.random.default_rng(seed)
    keys = np.array(keys)
    E, T, H = en.loc[keys, CATS].values, tr.loc[keys, CATS].values, human.loc[keys, CATS].values

    def mr(M, Hh):
        return np.mean([np.corrcoef(M[:, j], Hh[:, j])[0, 1] for j in range(4)])

    point = {"d_mean_r": mr(T, H) - mr(E, H)}
    for j, c in enumerate(CATS):
        point[f"d_share_{c}"] = T[:, j].mean() - E[:, j].mean()
    boots = {k: [] for k in point}
    n = len(keys)
    for _ in range(n_boot):
        idx = rng.integers(0, n, n)
        e, t, h = E[idx], T[idx], H[idx]
        boots["d_mean_r"].append(mr(t, h) - mr(e, h))
        for j, c in enumerate(CATS):
            boots[f"d_share_{c}"].append(t[:, j].mean() - e[:, j].mean())
    return {k: (point[k], np.percentile(boots[k], 2.5), np.percentile(boots[k], 97.5)) for k in point}


def within_turkish_did(en: pd.DataFrame, tr: pd.DataFrame, stim: pd.DataFrame,
                       n_boot: int = 2000, seed: int = 0) -> pd.DataFrame:
    """Difference-in-differences: does translation into a transparent native Turkish
    word shift the profile more than translation into an opaque loanword?

    For each concept, delta = Turkish share - English share (same model). Compare
    mean delta for native words with a visible concrete root against loanwords.
    Using the English profile of the same concept as the baseline controls for
    what the concept is, which a raw native-vs-loan comparison cannot.
    """
    rng = np.random.default_rng(seed)
    s = stim.set_index("english")
    keys = [k for k in tr.index.intersection(en.index) if k in s.index]
    d = (tr.loc[keys, CATS] - en.loc[keys, CATS])
    grp_a = [k for k in keys if s.loc[k, "origin"] == "native" and str(s.loc[k, "concrete_root"]) not in ("", "nan")]
    grp_b = [k for k in keys if str(s.loc[k, "origin"]).startswith("loan")]
    rows = []
    for c in CATS:
        a, b = d.loc[grp_a, c].values, d.loc[grp_b, c].values
        point = a.mean() - b.mean()
        bs = [rng.choice(a, len(a)).mean() - rng.choice(b, len(b)).mean() for _ in range(n_boot)]
        rows.append(dict(category=CAT_LABELS[c], native_transparent_shift=a.mean(), loanword_shift=b.mean(),
                         difference=point, ci_low=np.percentile(bs, 2.5), ci_high=np.percentile(bs, 97.5),
                         n_native=len(a), n_loan=len(b)))
    return pd.DataFrame(rows)


def cross_language_agreement(en: pd.DataFrame, tr: pd.DataFrame) -> Dict[str, float]:
    keys = en.index.intersection(tr.index)
    out = {c: stats.pearsonr(en.loc[keys, c], tr.loc[keys, c])[0] for c in CATS}
    out["mean"] = float(np.mean([out[c] for c in CATS]))
    return out


# ---------------------------------------------------------------------------
# Figure
# ---------------------------------------------------------------------------

def figure(human: pd.DataFrame, en: pd.DataFrame, tr: pd.DataFrame, keys: List[str],
           model_label: str, out_base: str, n_boot: int = 2000, seed: int = 0,
           r_en: Optional[float] = None, r_tr: Optional[float] = None):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    rng = np.random.default_rng(seed)
    keys = np.array(keys)
    series = [("Human norms (Harpaintner 2018)", human, "#8a8a8a"),
              (f"{model_label}, English", en, "#2f6db5"),
              (f"{model_label}, Turkish", tr, "#c8553d")]
    fig, ax = plt.subplots(figsize=(8.6, 4.6), dpi=200)
    width = 0.26
    x = np.arange(len(CATS))
    for i, (lab, df, col) in enumerate(series):
        M = df.loc[keys, CATS].values
        means = M.mean(0)
        bs = np.array([M[rng.integers(0, len(M), len(M))].mean(0) for _ in range(n_boot)])
        lo, hi = np.percentile(bs, 2.5, 0), np.percentile(bs, 97.5, 0)
        ax.bar(x + (i - 1) * width, means, width, color=col, label=lab,
               yerr=[means - lo, hi - means], capsize=3, error_kw=dict(lw=0.8, ecolor="#333"))
    ax.set_xticks(x, [CAT_LABELS[c] for c in CATS])
    ax.set_ylabel("Mean share of generated properties")
    ax.set_ylim(0, max(0.6, ax.get_ylim()[1]))
    ax.spines[["top", "right"]].set_visible(False)
    ax.legend(frameon=False, fontsize=8.5, loc="upper left")
    sub = f"{len(keys)} abstract concepts"
    if r_en is not None and r_tr is not None:
        sub += f"  |  Mean r with human norms: English {r_en:.3f}, Turkish {r_tr:.3f}"
    ax.set_title(f"Where the properties come from, English vs Turkish\n{sub}", fontsize=10.5)
    fig.text(0.01, 0.005, "Error bars: 95% bootstrap CI over concepts. Coder: gemini-2.5-flash-lite "
             "with the paper's English coding prompt in both arms.", fontsize=7, color="#555")
    fig.tight_layout(rect=(0, 0.03, 1, 1))
    fig.savefig(out_base + ".png")
    fig.savefig(out_base + ".pdf")
    plt.close(fig)
    return out_base + ".png"


# ---------------------------------------------------------------------------
# Correcting the released leaderboard
# ---------------------------------------------------------------------------

def _run_no(path: str) -> int:
    return int(re.search(r"_run_(\d+)\.csv$", path).group(1))


def recover_shipped(repo_exp_dir: str, model_dir: str) -> pd.DataFrame:
    """Rows of the released generations that the repo's parser read as empty but
    which contain properties in another format. One row per (run, word)."""
    import glob
    rows = []
    for p in sorted(glob.glob(os.path.join(repo_exp_dir, "generations", "exp1", f"{model_dir}_run_*.csv"))):
        g = pd.read_csv(p)
        for _, r in g.iterrows():
            try:
                orig = ast.literal_eval(str(r["properties"]))
            except Exception:
                orig = []
            if orig and any(str(x).strip() for x in orig):
                continue
            props, mode = parse_properties(str(r["response"]), str(r["word"]))
            rows.append(dict(run=_run_no(p), key=str(r["word"]).strip(), stimulus=str(r["word"]).strip(),
                             response=r["response"], properties=props, parse_mode=mode))
    return pd.DataFrame(rows)


def corrected_profile(repo_exp_dir: str, model_dir: str, recovered_coded: Dict[int, str]) -> Tuple[pd.DataFrame, Dict]:
    """Released coded runs with the empty-parse rows removed and, where available,
    replaced by freshly coded recovered properties.

    recovered_coded: {run number: path to a coded CSV produced by code()}.
    """
    import glob
    rows, dropped, added = [], 0, 0
    for p in sorted(glob.glob(os.path.join(repo_exp_dir, "coded_generations", "exp1", model_dir, "*_run_*.csv"))):
        run = _run_no(p)
        gen = pd.read_csv(os.path.join(repo_exp_dir, "generations", "exp1", f"{model_dir}_run_{run}.csv"))
        empty = set()
        for _, r in gen.iterrows():
            try:
                pr = ast.literal_eval(str(r["properties"]))
            except Exception:
                pr = []
            if not (pr and any(str(x).strip() for x in pr)):
                empty.add(str(r["word"]).strip())
        for _, r in pd.read_csv(p).iterrows():
            w = str(r["word"]).strip()
            if w in empty:
                dropped += 1
                continue
            try:
                fr = ast.literal_eval(str(r["frequencies"]))
            except Exception:
                continue
            if len(fr) == 4:
                rows.append(dict(run=run, key=w, **dict(zip(CATS, fr))))
        rp = recovered_coded.get(run)
        if rp and os.path.exists(rp):
            for _, r in pd.read_csv(rp).iterrows():
                try:
                    codes = ast.literal_eval(str(r["codes"]))
                    fr = ast.literal_eval(str(r["frequencies"]))
                except Exception:
                    continue
                if codes:
                    rows.append(dict(run=run, key=str(r["key"]), **dict(zip(CATS, fr))))
                    added += 1
    df = pd.DataFrame(rows)
    return df.groupby("key")[CATS].mean(), dict(dropped=dropped, added=added)


In [ ]:
# ---- Turkish stimuli ----
# The table below is the default. To use a corrected version, upload your edited
# stimuli_tr.csv into the output folder on Drive; it takes precedence.
import io, os, pandas as pd
DEFAULT_STIMULI = """english,turkish,alternative,origin,concrete_root,flag,note,reviewed
ability,kabiliyet,yeti,loan-arabic,,check,'yetenek' reserved for talent to avoid a duplicate target,
accusation,suçlama,,native,suç (crime),ok,,
achievement,kazanım,başarı,native,kazan- (earn),check,'başarı' reserved for success; the two collapse in Turkish,
activity,etkinlik,faaliyet,native,,check,reform-era coinage; 'faaliyet' is the older loan,
addiction,bağımlılık,,native,"bağ (tie, rope)",ok,,
admiration,hayranlık,,hybrid,,ok,,
advantage,avantaj,üstünlük,loan-french,,check,,
adventure,macera,,loan-arabic,,ok,,
advice,öğüt,nasihat,native,,check,'tavsiye' reserved for recommendation,
agony,ıstırap,,loan-arabic,,ok,,
agreement,anlaşma,mutabakat,native,anla- (understand),ok,,
amazement,hayret,,loan-arabic,,check,"cluster with astonishment, surprise, wonder",
ambition,hırs,tutku,loan-arabic,,check,'hırs' leans more negative than English 'ambition',
ambush,pusu,,native,,ok,,
amusement,eğlenme,eğlence,native,eğlen- (be diverted),check,reviewer changed from 'eğlence',2026-09-25 native speaker
anger,öfke,,native,,ok,,
anxiety,kaygı,endişe,native,,ok,,
apology,özür,,loan-arabic,,ok,,
appearance,görünüm,görünüş,native,gör- (see),ok,,
appetite,iştah,,loan-persian,,ok,Arabic via Persian,
application,başvuru,uygulama,native,baş (head) + vur- (strike),hard,English splits job application / applying a method; kept after native-speaker review,2026-09-25 native speaker
aptitude,yatkınlık,,native,"yat- (lie down, incline)",check,,
argument,tartışma,argüman,native,tart- (weigh),hard,quarrel vs logical argument; kept after native-speaker review,2026-09-25 native speaker
art,sanat,,loan-arabic,,ok,,
aspect,yön,boyut,native,,hard,no clean single-word equivalent; kept after native-speaker review,2026-09-25 native speaker
assassination,suikast,,loan-arabic,,ok,,
assumption,varsayım,,native,var (existing) + say- (count),ok,reform-era coinage,
astonishment,şaşkınlık,,native,şaş- (be confused),check,"cluster with amazement, surprise, wonder",
attack,saldırı,,native,saldır- (charge),ok,,
attention,dikkat,,loan-arabic,,ok,,
attraction,çekim,cazibe,native,çek- (pull),check,'çekim' is also physical/gravitational pull,
beauty,güzellik,,native,güzel (beautiful),ok,,
being,varlık,,native,var (existing),check,philosophical sense,
betrayal,ihanet,,loan-arabic,,ok,,
blackmail,şantaj,,loan-french,,ok,,
boredom,can sıkıntısı,bıkkınlık,hybrid,sık- (squeeze),check,"two-word form; 'can' is Persian, 'sık-' native",
boycott,boykot,,loan-other,,ok,,
bravery,yiğitlik,kahramanlık,native,yiğit (young brave man),check,'cesaret' reserved for courage,
career,kariyer,,loan-french,,ok,,
carefulness,özen,dikkatlilik,native,,check,,
catastrophe,felaket,afet,loan-arabic,,ok,,
caution,temkin,ihtiyat,loan-arabic,,check,,
challenge,meydan okuma,zorluk,hybrid,oku- (call out),hard,no single noun; 'zorluk' reserved for difficulty; kept after native-speaker review,2026-09-25 native speaker
chaos,kaos,,loan-other,,ok,Greek via French,
character,karakter,,loan-french,,ok,,
charm,çekicilik,cazibe,native,çek- (pull),check,,
comfort,rahatlık,konfor,hybrid,,ok,,
commitment,bağlılık,taahhüt,native,"bağ (tie, rope)",check,legal sense would be 'taahhüt',
comparison,karşılaştırma,,native,karşı (opposite),ok,,
compassion,şefkat,acıma,loan-arabic,,check,'merhamet' reserved for mercy,
competence,yetkinlik,,native,"yet- (reach, suffice)",ok,reform-era coinage,
compromise,uzlaşma,taviz,native,,ok,,
compulsion,zorlama,zorlantı,hybrid,,hard,reviewer changed from 'zorlantı',2026-09-25 native speaker
confession,itiraf,,loan-arabic,,ok,,
conflict,çatışma,,native,çat- (knock together),ok,,
consideration,değerlendirme,düşüncelilik,native,değer (value),hard,reviewer changed from 'düşüncelilik'; weighing a matter seriously (reviewer: 'bir konuyu ciddiyetle ele alıp tartma'),2026-09-25 native speaker
consolation,teselli,,loan-arabic,,ok,,
contempt,küçümseme,aşağılama,native,küçük (small),ok,,
convention,uzlaşım,teamül,native,,hard,"reviewer changed from 'teamül'; social convention; reviewer first chose 'gelenek', which is reserved for tradition",2026-09-25 native speaker
cooperation,işbirliği,,native,iş (work) + bir (one),ok,,
courage,cesaret,,loan-arabic,,ok,,
creativity,yaratıcılık,,native,yarat- (create),ok,,
crime,suç,,native,,ok,,
criticism,eleştiri,,native,ele- (sieve),ok,reform-era coinage,
cruelty,zulüm,acımasızlık,loan-arabic,,ok,,
culture,kültür,,loan-french,,ok,,
curiosity,merak,,loan-arabic,,ok,,
custom,âdet,görenek,loan-arabic,,check,'gelenek' reserved for tradition,
danger,tehlike,,loan-arabic,,ok,,
death,ölüm,,native,öl- (die),ok,,
decision,karar,,loan-arabic,,ok,,
demand,talep,istem,loan-arabic,,check,request vs economic demand,
desire,arzu,istek,loan-persian,,ok,,
desperation,çaresizlik,,hybrid,,ok,,
destruction,yıkım,,native,yık- (knock down),ok,,
detection,tespit,saptama,loan-arabic,,check,,
dictatorship,diktatörlük,,hybrid,,ok,,
difference,fark,ayrım,loan-arabic,,ok,,
difficulty,zorluk,,hybrid,,ok,'zor' is Persian,
dignity,haysiyet,saygınlık,loan-arabic,,check,'onur' reserved for honor,
diligence,çalışkanlık,,native,çalış- (work),ok,,
disadvantage,dezavantaj,,loan-french,,ok,,
disappointment,hayal kırıklığı,,hybrid,kır- (break),ok,"'hayal' Arabic, 'kır-' native",
discovery,keşif,,loan-arabic,,ok,,
disgust,iğrenme,tiksinti,native,iğren- (be repelled),ok,,
dismay,dehşet,yılgınlık,loan-arabic,,hard,reviewer changed from 'yılgınlık'; horror previously used dehşet; horror reassigned,2026-09-25 native speaker
dispute,anlaşmazlık,ihtilaf,native,anla- (understand),ok,,
distance,mesafe,uzaklık,loan-arabic,,ok,,
distraction,dikkat dağınıklığı,,hybrid,dağıl- (scatter),check,two-word form,
diversity,çeşitlilik,,hybrid,,ok,,
doubt,şüphe,kuşku,loan-arabic,,ok,,
downfall,çöküş,,native,çök- (collapse),ok,,
dream,rüya,hayal,loan-arabic,,hard,night dream vs aspiration; German 'Traum' has the same split; kept after native-speaker review,2026-09-25 native speaker
duty,vazife,ödev,loan-arabic,,check,'görev' reserved for task,
eagerness,heves,,loan-arabic,,check,,
effort,çaba,,native,,ok,,
elegance,zarafet,,loan-arabic,,ok,,
emotion,duygu,,native,"duy- (hear, feel)",ok,,
enchantment,büyülenme,,native,büyü (spell),check,,
enthusiasm,coşku,,native,coş- (boil over),ok,,
envy,haset,gıpta,loan-arabic,,check,'kıskançlık' reserved for jealousy,
error,hata,,loan-arabic,,ok,,
event,olay,,native,ol- (happen),ok,reform-era coinage,
excitement,heyecan,,loan-arabic,,ok,,
expectation,beklenti,,native,bekle- (wait),ok,,
experience,deneyim,tecrübe,native,dene- (try),ok,reform-era coinage,
explanation,açıklama,,native,açık (open),ok,,
exploitation,sömürü,,native,sömür- (gnaw),ok,,
faithfulness,sadakat,,loan-arabic,,ok,,
fantasy,fantezi,hayal gücü,loan-french,,check,,
fate,kader,,loan-arabic,,ok,,
fear,korku,,native,kork- (fear),ok,,
fight,kavga,dövüş,loan-persian,,ok,,
fitness,zindelik,form,hybrid,,check,,
flexibility,esneklik,,native,esne- (stretch),ok,,
force,kuvvet,,loan-arabic,,check,'güç' reserved for power,
forgiveness,bağışlama,affetme,native,bağışla- (grant),ok,,
freedom,özgürlük,hürriyet,native,öz (self),ok,,
fright,ürküntü,,native,ürk- (startle),check,"cluster with fear, horror, panic",
frustration,hüsran,engellenmişlik,loan-arabic,,hard,no close single-word equivalent; kept after native-speaker review,2026-09-25 native speaker
fun,eğlence,keyif,native,eğlen- (be diverted),hard,reviewer changed from 'keyif',2026-09-25 native speaker
fury,hiddet,,loan-arabic,,check,"cluster with anger, rage",
future,gelecek,istikbal,native,gel- (come),ok,,
genius,deha,,loan-arabic,,check,"quality, not the person ('dâhi')",
glory,şan,şeref,loan-arabic,,check,,
grace,incelik,lütuf,native,"ince (thin, fine)",hard,elegance vs divine grace; 'zarafet' reserved for elegance; kept after native-speaker review,2026-09-25 native speaker
gratitude,minnettarlık,şükran,hybrid,,ok,,
greed,açgözlülük,,native,aç (hungry) + göz (eye),ok,,
guess,tahmin,,loan-arabic,,ok,,
habit,alışkanlık,,native,alış- (get used to),ok,,
happiness,mutluluk,,native,,check,origin of 'mutlu' should be confirmed,
hate,nefret,,loan-arabic,,ok,,
help,yardım,,native,,check,origin should be confirmed,
hint,ipucu,,native,ip (thread) + uç (end),ok,,
home,ev,yuva,native,ev (house),hard,reviewer changed from 'yuva',2026-09-25 native speaker
honor,onur,şeref,loan-french,,ok,from French honneur,
hope,umut,ümit,native,,ok,,
horror,yoğun korku,dehşet,native,kork- (fear),ok,reviewer changed from 'dehşet'; reviewer typed 'yoğun kokru'; read as typo for 'yoğun korku'. Freed 'dehşet' for dismay; 'korku' alone stays with fear,2026-09-25 native speaker
humor,mizah,,loan-arabic,,ok,,
hunger,açlık,,native,aç (hungry),ok,,
hurry,acele,telaş,loan-arabic,,ok,,
idea,fikir,,loan-arabic,,ok,,
ideal,ülkü,ideal,native,,check,'ülkü' is dated and politically coloured; 'ideal' is the everyday word,
illusion,yanılsama,illüzyon,native,yanıl- (err),ok,reform-era coinage,
impression,izlenim,,native,"iz (footprint, trace)",ok,reform-era coinage,
incentive,teşvik,özendirme,loan-arabic,,ok,,
independence,bağımsızlık,,native,"bağ (tie, rope)",ok,,
indifference,kayıtsızlık,ilgisizlik,hybrid,,ok,,
influence,etki,,native,et- (do),ok,reform-era coinage,
insanity,delilik,,native,deli (mad),ok,,
insight,içgörü,,native,iç (inside) + gör- (see),ok,reform-era calque,
insult,hakaret,,loan-arabic,,ok,,
insurrection,ayaklanma,,native,ayak (foot),ok,,
intention,niyet,,loan-arabic,,ok,,
interest,ilgi,,native,,check,not financial interest ('faiz'),
invention,buluş,icat,native,bul- (find),ok,,
jealousy,kıskançlık,,native,kıskan- (begrudge),ok,,
joy,sevinç,,native,sevin- (rejoice),ok,,
judgement,yargı,,native,yar- (split),ok,,
justice,adalet,,loan-arabic,,ok,,
justification,gerekçe,,native,gerek (need),ok,,
kindness,iyilik,nezaket,native,iyi (good),check,'iyilik' leans toward goodness or a good deed,
knowledge,bilgi,,native,bil- (know),ok,,
love,sevgi,aşk,native,sev- (love),check,'aşk' is romantic love,
luck,şans,talih,loan-french,,ok,,
lust,şehvet,,loan-arabic,,ok,,
luxury,lüks,,loan-french,,ok,,
magic,büyü,,native,,ok,,
manipulation,manipülasyon,yönlendirme,loan-french,,ok,,
manners,görgü,terbiye,native,gör- (see),ok,,
matter,madde,mesele,loan-arabic,,hard,reviewer changed from 'mesele'; physical matter,2026-09-25 native speaker
meanness,alçaklık,cimrilik,native,alçak (low),check,baseness vs stinginess,
memory,anı,hafıza,native,an- (recall),hard,a memory vs the faculty; kept after native-speaker review,2026-09-25 native speaker
menace,gözdağı,,native,göz (eye) + dağ (brand),check,'tehdit' reserved for threat,
mercy,merhamet,,loan-arabic,,ok,,
miracle,mucize,,loan-arabic,,ok,,
misdemeanor,kabahat,,loan-arabic,,ok,,
misery,sefalet,,loan-arabic,,check,'sefalet' leans toward squalor,
mockery,alay,,loan-arabic,,check,origin should be confirmed,
model,model,örnek,loan-french,,hard,role model vs fashion model vs scientific model; kept after native-speaker review,2026-09-25 native speaker
moment,an,,native,,ok,,
mood,ruh hali,keyif,loan-arabic,,check,two-word form,
mourning,yas,,loan-persian,,ok,,
need,ihtiyaç,gereksinim,loan-arabic,,ok,,
nervousness,gerginlik,sinirlilik,native,ger- (stretch taut),ok,,
nightmare,kâbus,,loan-arabic,,ok,,
notion,kavram,,native,kavra- (grasp),ok,reform-era coinage,
obedience,itaat,,loan-arabic,,ok,,
objection,itiraz,,loan-arabic,,ok,,
obligation,yükümlülük,zorunluluk,native,yük (load),ok,,
observation,gözlem,,native,göz (eye),ok,reform-era coinage,
occupation,meslek,işgal,loan-arabic,,hard,job vs military occupation; kept after native-speaker review,2026-09-25 native speaker
order,düzen,emir,native,"düz (flat, straight)",hard,orderliness vs command vs sequence; kept after native-speaker review,2026-09-25 native speaker
ostracism,dışlanma,,native,dış (outside),ok,,
overview,genel bakış,,native,bak- (look),ok,two-word form,
pain,acı,,native,,ok,,
panic,panik,,loan-french,,ok,,
pardon,af,,loan-arabic,,ok,,
passion,tutku,,native,tut- (hold),ok,,
past,geçmiş,,native,geç- (pass),ok,,
peace,barış,,native,,ok,,
perception,algı,,native,al- (take),ok,reform-era coinage,
perfection,mükemmellik,kusursuzluk,hybrid,,ok,,
performance,performans,başarım,loan-french,,check,,
perspective,bakış açısı,,native,bak- (look) + açı (angle),ok,two-word form,
plan,plan,,loan-french,,ok,,
pleasure,zevk,,loan-arabic,,ok,,
potential,potansiyel,gizil güç,loan-french,,ok,,
poverty,yoksulluk,,native,yok (absent),ok,,
power,güç,,native,,ok,,
prediction,öngörü,tahmin,native,ön (front) + gör- (see),ok,reform-era calque; 'tahmin' reserved for guess,
present,şimdiki zaman,şimdi,native,,hard,time sense only; English also means gift; kept after native-speaker review,2026-09-25 native speaker
pride,gurur,,loan-arabic,,ok,,
problem,sorun,problem,native,sor- (ask),ok,reform-era coinage,
prohibition,yasak,,native,,ok,,
promise,vaat,söz,loan-arabic,,check,'söz' is more natural but also means 'word',
proof,kanıt,,native,,ok,reform-era coinage,
proposal,öneri,,native,ön (front),ok,reform-era coinage,
prosperity,refah,,loan-arabic,,ok,,
protection,koruma,,native,koru- (guard),ok,,
protest,protesto,,loan-other,,ok,Italian,
provocation,provokasyon,kışkırtma,loan-french,,check,,
punishment,ceza,,loan-arabic,,ok,,
quality,nitelik,kalite,native,,ok,reform-era coinage,
quantity,nicelik,miktar,native,,ok,reform-era coinage,
rage,hışım,,loan-persian,,check,"cluster with anger, fury",
realization,farkına varma,farkındalık,hybrid,var- (arrive),hard,reviewer changed from 'farkındalık',2026-09-25 native speaker
reassurance,güvence,,native,güven (trust),ok,,
recommendation,tavsiye,öneri,loan-arabic,,ok,,
recovery,iyileşme,,native,iyi (good),ok,,
regard,saymak,itibar,native,"say- (count, esteem)",hard,"reviewer changed from 'itibar'; reviewer's choice; a verb infinitive, unlike most stimuli; shares its root with saygı (respect)",2026-09-25 native speaker
regret,pişmanlık,,hybrid,,ok,,
relief,rahatlama,,hybrid,,ok,,
remorse,vicdan azabı,,loan-arabic,,check,two-word form,
requirement,gereklilik,gereksinim,native,gerek (need),ok,,
resemblance,benzerlik,,native,benze- (resemble),ok,,
resistance,direniş,direnç,native,diren- (resist),ok,,
resolution,önerge,çözüm,native,ön (front),hard,reviewer changed from 'çözüm'; a formal resolution or motion put to an assembly,2026-09-25 native speaker
respect,saygı,,native,"say- (count, esteem)",ok,,
responsibility,sorumluluk,,native,sor- (ask),ok,,
retaliation,misilleme,,hybrid,,ok,,
revenge,intikam,,loan-arabic,,ok,,
revolution,devrim,ihtilal,hybrid,,check,reform-era coinage on an Arabic root,
risk,risk,,loan-french,,ok,,
ritual,ritüel,ayin,loan-french,,ok,,
romantic,romantiklik,romantizm,hybrid,,hard,English stimulus is an adjective; kept after native-speaker review,2026-09-25 native speaker
safety,güvenlik,,native,güven (trust),ok,,
satisfaction,tatmin,memnuniyet,loan-arabic,,ok,,
scheme,tasarı,entrika,native,,hard,"reviewer changed from 'entrika'; a plan or design; reviewer first chose 'plan', which is the stimulus plan's own translation",2026-09-25 native speaker
selection,seçim,,native,seç- (choose),check,also means election,
self-dependence,kendine yeterlilik,özgüven,native,yet- (suffice),hard,no settled equivalent; two-word form; kept after native-speaker review,2026-09-25 native speaker
sensation,duyum,his,native,"duy- (hear, feel)",hard,bodily sensation vs sensational event; kept after native-speaker review,2026-09-25 native speaker
sensitivity,duyarlılık,,native,"duy- (hear, feel)",ok,,
shame,utanç,,native,utan- (be ashamed),ok,,
shyness,çekingenlik,utangaçlık,native,çekin- (draw back),ok,,
skill,beceri,,native,becer- (manage),ok,,
slapstick,şaklabanlık,kaba güldürü,hybrid,,hard,"reviewer changed from 'kaba güldürü'; origin of şaklaban uncertain, check Nişanyan",2026-09-25 native speaker
solitude,yalnızlık,,native,yalnız (alone),ok,,
stability,istikrar,kararlılık,loan-arabic,,ok,,
stimulus,uyaran,uyarıcı,native,uyar- (rouse),check,shares a root with warning,
stress,stres,,loan-french,,ok,,
success,başarı,,native,baş (head),ok,,
suffering,çile,acı çekme,loan-persian,,check,'ıstırap' reserved for agony,
support,destek,,native,,check,origin should be confirmed,
surprise,sürpriz,şaşırma,loan-french,,check,"cluster with amazement, astonishment, wonder",
talent,yetenek,,native,"yet- (reach, suffice)",ok,,
task,görev,,native,,ok,,
terrorism,terörizm,,loan-french,,ok,,
theory,kuram,teori,native,kur- (build),ok,reform-era coinage,
thirst,susuzluk,,native,su (water),ok,,
thought,düşünce,,native,düşün- (think),ok,,
threat,tehdit,,loan-arabic,,ok,,
tradition,gelenek,,native,gel- (come),ok,,
trend,eğilim,trend,native,eğil- (bend),ok,,
trust,güven,,native,,ok,,
value,değer,,native,değ- (touch),ok,,
vanity,kendini beğenmişlik,kibir,native,beğen- (like),check,two-word form,
victory,zafer,,loan-arabic,,ok,,
view,görüş,manzara,native,gör- (see),check,opinion vs scenery,
violation,ihlal,,loan-arabic,,ok,,
violence,şiddet,,loan-arabic,,ok,,
war,savaş,,native,,ok,,
warning,uyarı,,native,uyar- (rouse),ok,,
weakness,zayıflık,güçsüzlük,hybrid,,ok,,
wealth,zenginlik,servet,hybrid,,ok,,
welfare,esenlik,refah,native,esen (well),hard,'refah' reserved for prosperity; kept after native-speaker review,2026-09-25 native speaker
will,irade,,loan-arabic,,ok,,
willingness,isteklilik,,native,iste- (want),ok,,
wish,dilek,,native,dile- (wish),ok,,
wit,nükte,espri,loan-arabic,,check,,
wonder,merak etmek,huşu,hybrid,,hard,"reviewer changed from 'huşu'; verb phrase 'to wonder'; distinct from curiosity, which keeps 'merak'",2026-09-25 native speaker
work,iş,emek,native,,check,,
worry,endişe,,loan-persian,,ok,,
"""
path = os.path.join(OUT, "stimuli_tr.csv")
if os.path.exists(path):
    stim = pd.read_csv(path, keep_default_na=False); print("using your edited", path)
else:
    stim = pd.read_csv(io.StringIO(DEFAULT_STIMULI), keep_default_na=False)
    stim.to_csv(path, index=False); print("wrote default table to", path, "(edit it there if you change translations)")
print(stim.flag.value_counts().to_dict(), "|", stim.origin.value_counts().to_dict())
stim.head(8)

### The smallest change to the original repo

The repo calls every model through OpenRouter. Gemini exposes an OpenAI-compatible endpoint, so pointing the repo at it is one new client file plus one registry line. The cell below writes both into the cloned repo, so the original `generate.py` and `code.py` also work with `--client gemini`. This notebook uses its own resumable runner, but it calls the same endpoint.

In [ ]:
gemini_client = '''"""Gemini client via Google's OpenAI-compatible endpoint.

Set GEMINI_API_KEY in the environment. Use model ids without a provider prefix,
e.g. gemini-2.5-flash-lite.
"""
import os
from openai import OpenAI

_client = None


def _get_client():
    global _client
    if _client is None:
        _client = OpenAI(api_key=os.environ["GEMINI_API_KEY"],
                         base_url="https://generativelanguage.googleapis.com/v1beta/openai/")
    return _client


def query_llm(prompt, model_name, seed=None, temperature=1.0, max_tokens=512):
    model_name = model_name.split("/", 1)[-1]  # accept google/gemini-... too
    r = _get_client().chat.completions.create(
        model=model_name, messages=[{"role": "user", "content": prompt}],
        temperature=temperature, max_tokens=max_tokens)
    return r.choices[0].message.content or ""
'''
p = f"{REPO}/src/llm_clients/gemini.py"
open(p, "w").write(gemini_client)
reg = f"{REPO}/src/llm_clients/__init__.py"
s = open(reg).read()
if '"gemini"' not in s:
    s = s.replace('"local": "local",', '"local": "local",\n    "gemini": "gemini",')
    open(reg, "w").write(s)
os.environ["GEMINI_API_KEY"] = API_KEY
print(open(reg).read().split("REGISTRY")[1][:120])

In [ ]:
# ---- smoke test: one call per arm, printed raw ----
sys.path.insert(0, "/content")
import importlib, gg_tr as G; importlib.reload(G)
query = G.make_gemini_query(API_KEY, rpm=RPM)
avail = [m.id for m in query.client.models.list()]
print("flash-lite models on this key:", [m for m in avail if "flash-lite" in m])
assert any(MODEL in m for m in avail), f"{MODEL} is not available on this key; change MODEL"

EN_GEN = open(f"{REPO}/data/experiment_1/generation_prompt.txt").read()
EN_CODE = open(f"{REPO}/data/experiment_1/coding_prompt.txt").read()
for word, prompt in [("justice", EN_GEN), ("adalet", G.TR_GENERATION_PROMPT)]:
    r = query(prompt.replace("{word}", word), model_name=MODEL)
    props, mode = G.parse_properties(r, word)
    c = query(EN_CODE.replace("{word}", word).replace("{properties}", ", ".join(props)), model_name=CODER, temperature=0.0)
    print(f"--- {word} ---\n{r}\nparsed ({mode}): {props}\ncoder:\n{c}\ncodes: {G.parse_codes(c, G.EN_LABEL_MAP)[0]}\n")

## 1. Generate and code, both languages
Pilot first with `SUBSET_N = 30`. When the numbers look sane, set `SUBSET_N = None` and run this cell and everything below again.

In [ ]:
rows = stim if SUBSET_N is None else stim.sample(SUBSET_N, random_state=SEED)
EN = list(zip(rows.english, rows.english))
TR = list(zip(rows.english, rows.turkish))
tag = "all" if SUBSET_N is None else f"sub{SUBSET_N}"
calls = len(rows) * 2 * 2 * N_RUNS
print(f"{len(rows)} words x 2 languages x (generate + code) x {N_RUNS} runs = up to {calls} calls")
print(f"at {RPM} requests/min that is about {calls / RPM / 60:.1f} h (finished work is skipped)")

for run in range(1, N_RUNS + 1):
    for lang, items, prompt in [("en", EN, EN_GEN), ("tr", TR, G.TR_GENERATION_PROMPT)]:
        gp = f"{OUT}/{MODEL}/gen/{lang}_run_{run}.csv"
        cp = f"{OUT}/{MODEL}/coded/{lang}_run_{run}.csv"
        G.generate(items, prompt, MODEL, gp, query, concurrency=CONCURRENCY)
        G.code(gp, EN_CODE, CODER, cp, query, G.EN_LABEL_MAP, concurrency=CONCURRENCY)
u = query.usage
print(f"calls this session {u.calls}, tokens in {u.prompt_tokens:,} out {u.completion_tokens:,}, "
      f"cost if billed about ${u.cost():.3f}")

## 2. Check the raw outputs before trusting any number
Format drift between languages is exactly how the released leaderboard lost a quarter of Flash-Lite's answers (section 7). This shows how each language's answers were parsed, and a sample of Turkish outputs for a native speaker to eyeball.

In [ ]:
import glob
for lang in ["en", "tr"]:
    g = pd.concat([pd.read_csv(p) for p in sorted(glob.glob(f"{OUT}/{MODEL}/gen/{lang}_run_*.csv"))])
    print(lang, "parse modes:", g.parse_mode.value_counts().to_dict())
g = pd.concat([pd.read_csv(p) for p in sorted(glob.glob(f"{OUT}/{MODEL}/gen/tr_run_*.csv"))])
for _, r in g.sample(min(15, len(g)), random_state=SEED).iterrows():
    print(f"{r.stimulus:22s} -> {r.properties}")

## 3. Results

In [ ]:
import numpy as np
human = pd.read_csv(f"{REPO}/data/experiment_1/human_norms.csv").rename(columns={"IS/E": "IS_E"})
human["word"] = human.word.astype(str).str.strip(); H = human.set_index("word")
en = G.profile(G.load_coded(glob.glob(f"{OUT}/{MODEL}/coded/en_run_*.csv")))
tr = G.profile(G.load_coded(glob.glob(f"{OUT}/{MODEL}/coded/tr_run_*.csv")))
keys = sorted(set(en.index) & set(tr.index) & set(H.index))
r_en, r_tr = G.mean_r(en, H, keys), G.mean_r(tr, H, keys)

print(f"words with both languages: {len(keys)}  runs: {N_RUNS}")
print(f"Mean r with human norms   English {r_en['mean']:.3f}   Turkish {r_tr['mean']:.3f}")
for c in G.CATS:
    print(f"   {G.CAT_LABELS[c]:20s} EN {r_en[c]:+.3f}   TR {r_tr[c]:+.3f}")

shipped = G.load_shipped_profile(REPO, SHIPPED_DIR, as_released=False)
if shipped is not None:
    rel = G.load_shipped_profile(REPO, SHIPPED_DIR, as_released=True)
    k2 = sorted(set(shipped.index) & set(en.index))
    print(f"\nsanity: our English vs the paper's released {SHIPPED_DIR} runs, profile agreement r = "
          f"{G.cross_language_agreement(shipped.loc[k2], en.loc[k2])['mean']:.3f}")
    print(f"        paper's released runs on these words: Mean r {G.mean_r(rel, H, k2)['mean']:.3f} as released, "
          f"{G.mean_r(shipped, H, k2)['mean']:.3f} with unparsed answers dropped (see section 7)")

agree = G.cross_language_agreement(en.loc[keys], tr.loc[keys])
print(f"\nsame model, English vs Turkish profile agreement: mean r = {agree['mean']:.3f}")
print("(how much of each concept's profile survives translation; the model's human alignment is the number above)")

In [ ]:
boot = G.paired_bootstrap(en, tr, H, keys, n_boot=2000, seed=SEED)
print("Turkish minus English, 95% bootstrap CI over words:")
for k, (p, lo, hi) in boot.items():
    lab = "Mean r with humans" if k == "d_mean_r" else "share of " + G.CAT_LABELS[k.split("_", 2)[2]]
    print(f"  {lab:32s} {p:+.3f}  [{lo:+.3f}, {hi:+.3f}]")

ok = [k for k in keys if stim.set_index("english").loc[k, "flag"] == "ok"]
b2 = G.paired_bootstrap(en, tr, H, ok, n_boot=2000, seed=SEED)
p, lo, hi = b2["d_mean_r"]
print(f"\nrobustness, only the {len(ok)} confidently translated words: delta Mean r {p:+.3f} [{lo:+.3f}, {hi:+.3f}]")

### Within Turkish: transparent native words vs loanwords
For each concept, shift = Turkish share minus English share for the same model. The table compares the average shift for native words with a visible concrete root against loanwords. A positive Sensorimotor difference would mean a visible concrete root pulls the model toward bodily properties, which English cannot show because its abstract vocabulary is mostly opaque Latinate borrowing.

In [ ]:
did = G.within_turkish_did(en.loc[keys], tr.loc[keys], stim, n_boot=2000, seed=SEED)
did.round(3)

In [ ]:
png = G.figure(H, en, tr, keys, MODEL, f"{OUT}/{MODEL}/fig_en_vs_tr", n_boot=2000, seed=SEED,
               r_en=r_en["mean"], r_tr=r_tr["mean"])
from IPython.display import Image; Image(png)

## 4. The numbers for the email

In [ ]:
import json
p, lo, hi = boot["d_mean_r"]
sm = did.set_index("category").loc["Sensorimotor"]
summary = f"""On {len(keys)} abstract concepts, {N_RUNS} runs per language, {MODEL} generating and coding:
  Mean r with the Harpaintner human norms: English {r_en['mean']:.3f}, Turkish {r_tr['mean']:.3f}
  Turkish minus English: {p:+.3f} (95% CI {lo:+.3f} to {hi:+.3f})
  Internal-state share: English {en.loc[keys,'IS_E'].mean():.3f}, Turkish {tr.loc[keys,'IS_E'].mean():.3f}
  Verbal-association share: English {en.loc[keys,'VA'].mean():.3f}, Turkish {tr.loc[keys,'VA'].mean():.3f}
  Transparent native vs loanword, Sensorimotor shift difference: {sm.difference:+.3f} (95% CI {sm.ci_low:+.3f} to {sm.ci_high:+.3f})"""
print(summary)
res = dict(model=MODEL, coder=CODER, n_runs=N_RUNS, n_words=len(keys), subset=SUBSET_N,
           mean_r_en=r_en, mean_r_tr=r_tr, cross_language_agreement=agree,
           bootstrap_tr_minus_en={k: dict(point=v[0], ci_low=v[1], ci_high=v[2]) for k, v in boot.items()},
           bootstrap_ok_words_only=dict(n=len(ok), point=b2["d_mean_r"][0], ci_low=b2["d_mean_r"][1], ci_high=b2["d_mean_r"][2]),
           within_turkish=did.to_dict("records"))
os.makedirs(f"{OUT}/{MODEL}", exist_ok=True)
open(f"{OUT}/{MODEL}/summary_{tag}.txt", "w").write(summary + "\n")
json.dump(res, open(f"{OUT}/{MODEL}/results_{tag}.json", "w"), indent=2, default=float)
did.to_csv(f"{OUT}/{MODEL}/within_turkish_{tag}.csv", index=False)
print(f"\nsaved summary_{tag}.txt, results_{tag}.json and within_turkish_{tag}.csv in {OUT}/{MODEL}")

## 5. Limitations to state before anyone else does
- **No Turkish human norms.** The human side of both comparisons is German speakers (Harpaintner et al. 2018). English is already a translation of their stimuli, so the Turkish arm is no less legitimate than the English one, but a difference between them is a difference in the model, not proof that the human-model gap differs.
- **Translation.** The Turkish list was translated from the English list, not from the original German, and 29 items have no clean one-to-one equivalent (`flag = hard`). The robustness line in section 3 reruns the main comparison on confidently translated items only.
- **Origins** (native, hybrid, loan) are a first pass and should be checked against Nişanyan Sözlüğü before the within-Turkish contrast is reported.
- **Reform-era coinages.** Many transparent native words (*içgörü*, *öngörü*, *kavram*, *izlenim*) were coined in the 20th-century language reform. Transparency and recency are confounded.
- **Coder.** Turkish properties are coded with the English prompt by a model reading Turkish. Section 6 reruns the coding with a Turkish prompt as a check.
- **Runs.** 3 runs, not the paper's 10.

## 6. Optional robustness: code the Turkish properties with a Turkish coding prompt

In [ ]:
for run in range(1, N_RUNS + 1):
    G.code(f"{OUT}/{MODEL}/gen/tr_run_{run}.csv", G.TR_CODING_PROMPT, CODER,
           f"{OUT}/{MODEL}/coded_trprompt/tr_run_{run}.csv", query, G.TR_LABEL_MAP, concurrency=CONCURRENCY)
trp = G.profile(G.load_coded(glob.glob(f"{OUT}/{MODEL}/coded_trprompt/tr_run_*.csv")))
kk = sorted(set(keys) & set(trp.index))
print(f"Turkish, Turkish coding prompt: Mean r = {G.mean_r(trp, H, kk)['mean']:.3f} (English prompt: {G.mean_r(tr, H, kk)['mean']:.3f})")
print("agreement between the two codings of the same Turkish properties:",
      round(G.cross_language_agreement(tr.loc[kk], trp.loc[kk])['mean'], 3))

## 7. Optional: correcting the released English leaderboard

The repo's parser only accepts answers that contain the literal label `properties:`. When a model answers `anger: shouting, red face, clenched fists, frustration` instead, the parser returns nothing, and `evaluate.py` averages that word's run in as an all-zero profile. In the released data this affects 25% of Gemini 2.5 Flash-Lite's word-runs, 41% of Llama 3.1 8B's and 17% of GPT-OSS 120B's. Simply dropping those rows (no API calls) moves Flash-Lite from 0.348 to 0.394 and Llama 3.1 8B from 0.283 to 0.339.

This cell does it properly: recovers the properties with a format-tolerant parser, codes them with the paper's coder, and recomputes. It is the only way to get the exact corrected numbers, and it needs Gemini 2.5 Flash-Lite, so it too has to run before the model retires.

In [ ]:
AFFECTED = ["gemini-2.5-flash-lite", "gemini-2.5-flash", "gpt-oss-120b", "llama-3.1-8b"]
rows_out = []
for m in AFFECTED:
    rec = G.recover_shipped(REPO, m)
    rec = rec[rec.properties.apply(len) > 0]
    paths = {}
    for run, grp in rec.groupby("run"):
        gp = f"{OUT}/corrected/{m}/gen_run_{run}.csv"
        os.makedirs(os.path.dirname(gp), exist_ok=True)
        grp[["key", "stimulus", "response", "properties", "parse_mode"]].to_csv(gp, index=False)
        cp = f"{OUT}/corrected/{m}/coded_run_{run}.csv"
        G.code(gp, EN_CODE, CODER, cp, query, G.EN_LABEL_MAP, concurrency=CONCURRENCY)
        paths[run] = cp
    released = G.load_shipped_profile(REPO, m, as_released=True)
    dropped, _ = G.corrected_profile(REPO, m, {})
    fixed, info = G.corrected_profile(REPO, m, paths)
    rows_out.append(dict(model=m, released=G.mean_r(released, H)["mean"], empties_dropped=G.mean_r(dropped, H)["mean"],
                         recovered_and_recoded=G.mean_r(fixed, H)["mean"], **info))
corr = pd.DataFrame(rows_out).round(3); corr.to_csv(f"{OUT}/corrected_leaderboard.csv", index=False); corr

In [ ]:
# ---- bundle everything for download ----
import shutil
z = shutil.make_archive("/content/grounding_gap_tr_results", "zip", OUT)
print(z)
try:
    from google.colab import files; files.download(z)
except Exception:
    pass